# Forecasting, Market Data, and Scenario Notebook

This notebook validates the forecasting and market-data pipeline on the cleaned Scherzinger data before the logic is promoted further into production workflows.

Primary outputs are written to `notebooks/output/`:
- `internal_timeseries.parquet`
- `market_series.parquet`
- `market_series_catalog.csv`
- `correlation_map.json`
- `forecast_baseline.json`
- `validation_report.md`

## 1. Setup

Load the tested helper module, check local secrets, and create the output directory.

In [8]:
from pathlib import Path
import json
import sys

import pandas as pd
from dotenv import dotenv_values

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
PLATFORM_ROOT = REPO_ROOT / "scherzinger-platform"
if str(PLATFORM_ROOT) not in sys.path:
    sys.path.insert(0, str(PLATFORM_ROOT))

from backend.services.forecasting_notebook import (
    build_correlation_map,
    build_forecast_baseline,
    build_default_market_configs,
    build_internal_timeseries,
    build_market_series_catalog,
    default_market_fetchers,
    ensure_output_dir,
    fetch_market_series,
    load_internal_datasets,
    load_required_env_keys,
    summarize_internal_data_audit,
    summarize_invoice_data_quality,
    write_json_artifact,
    write_market_artifacts,
)

OUTPUT_DIR = ensure_output_dir(REPO_ROOT / "notebooks" / "output")
ENV_PATH = REPO_ROOT / ".env"
required_keys = ["FRED_API_KEY", "EIA_API_KEY"]
env_status = load_required_env_keys(ENV_PATH, required_keys)
env_values = dotenv_values(ENV_PATH)

print(f"Repo root: {REPO_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
pd.DataFrame.from_dict(env_status, orient="index")

Repo root: /Users/dharmendersingh/Documents/Scherzinger_new
Output dir: /Users/dharmendersingh/Documents/Scherzinger_new/notebooks/output


,present,masked
FRED_API_KEY,True,6dbc...72a5
EIA_API_KEY,True,0W5R...pDXU


## 2. Load Internal Data

Read the cleaned parquet files and summarize the row counts, date ranges, and invoice data-quality flags.

In [9]:
datasets = load_internal_datasets(REPO_ROOT / "Data" / "cleaned")
audit = summarize_internal_data_audit(datasets)
dq_summary = summarize_invoice_data_quality(datasets["invoices_df"])

display(pd.DataFrame.from_dict(audit, orient="index"))
display(pd.DataFrame({"count": dq_summary["counts"], "share_pct": dq_summary["shares"]}))

,row_count,date_min,date_max
invoices_df,5565,2022-01-10,2025-12-17
quotes_df,4539,2022-01-04,2025-12-23
customers_df,1438,2022-01-04,2025-12-23
products_df,1798,None,None


,count,share_pct
dq_missing_margin,20,0.36
dq_negative_margin,13,0.23
dq_low_margin,20,0.36
dq_any_issue,53,0.95


## 3. Internal Monthly Time Series

Aggregate invoices into monthly `total`, `commodity_group`, and `business_unit` time series.

In [10]:
internal_timeseries = build_internal_timeseries(datasets["invoices_df"])
internal_path = OUTPUT_DIR / "internal_timeseries.parquet"
internal_timeseries.to_parquet(internal_path, index=False)

print(f"Wrote {internal_path}")
display(internal_timeseries.groupby(["grain", "metric"]).size().unstack(fill_value=0))
display(internal_timeseries.head())

Wrote /Users/dharmendersingh/Documents/Scherzinger_new/notebooks/output/internal_timeseries.parquet


metric,db1_margin,db2_margin,hkvar_per_unit,material_per_unit,revenue,volume
grain,,,,,,
business_unit,48,48,48,48,48,48
commodity_group,168,168,168,168,168,168
total,48,48,48,48,48,48


,grain,key,ts,metric,value
0,business_unit,BU001,2022-01-31,db1_margin,0.679855
1,business_unit,BU001,2022-01-31,db2_margin,0.608783
2,business_unit,BU001,2022-01-31,hkvar_per_unit,214.551560
3,business_unit,BU001,2022-01-31,material_per_unit,112.476119
4,business_unit,BU001,2022-01-31,revenue,493915.870000


## 4. Market Data Fetch

Fetch FRED, ECB, and EIA series into one normalized market-data table. Provider failures are captured in `market_status` so one unavailable source does not abort the notebook.

In [11]:
market_configs = build_default_market_configs(start_period="2018-01", end_period="2026-12")
market_fetchers = default_market_fetchers(
    fred_api_key=env_values.get("FRED_API_KEY", ""),
    eia_api_key=env_values.get("EIA_API_KEY", ""),
)
market_series, market_status = fetch_market_series(market_configs, fetchers=market_fetchers)
market_paths = write_market_artifacts(market_series, output_dir=OUTPUT_DIR)
market_catalog = build_market_series_catalog(market_series)

print(f"Wrote {market_paths['market_series_parquet']}")
print(f"Wrote {market_paths['market_series_catalog_csv']}")
display(pd.DataFrame(market_status))
display(market_catalog)

Wrote /Users/dharmendersingh/Documents/Scherzinger_new/notebooks/output/market_series.parquet
Wrote /Users/dharmendersingh/Documents/Scherzinger_new/notebooks/output/market_series_catalog.csv


/Users/dharmendersingh/Documents/Scherzinger_new/scherzinger-platform/backend/services/forecasting_notebook.py:424: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(frames, ignore_index=True).sort_values(["series_id", "ts"]).reset_index(drop=True)


,provider,series_id,ok,rows,error
0,fred,WPU101,True,99,None
1,fred,WPU114,True,99,None
2,fred,WPU114907,True,85,None
3,fred,PCOPPUSDM,True,99,None
4,fred,PALUMUSDM,True,99,None
5,fred,DEXUSEU,True,101,None
6,fred,DEUPROINDMISMEI,True,75,None
7,fred,DEUPROMANMISMEI,True,73,None
8,fred,DEUPRINTO01GYSAM,True,72,None
9,fred,DEUPRMNTO01GPSAM,True,73,None


,series_id,source,name,unit,freq,first_ts,last_ts,row_count,missing_pct
0,CPIAUCSL,fred,Consumer Price Index for All Urban Consumers: ...,Index 1982-1984=100,M,2018-01-31,2026-04-30,100,1.00
1,DCOILBRENTEU,fred,Crude Oil Prices: Brent - Europe,Dollars per Barrel,M,2018-01-31,2026-05-31,101,0.00
2,DEUPPDMMINMEI,fred,Producer Prices Index: Economic Activities: Ma...,Index 2015=100,M,2018-01-31,2022-12-31,60,0.00
3,DEUPRINTO01GYSAM,fred,Production: Industry: Total Industry Excluding...,Growth rate same period previous year,M,2018-01-31,2023-12-31,72,0.00
4,DEUPRMNTO01GPSAM,fred,Production: Manufacturing: Total Manufacturing...,Growth rate previous period,M,2018-01-31,2024-01-31,73,0.00
5,DEUPROINDMISMEI,fred,"Production, Sales, Work Started and Orders: Pr...",Index 2015=100,M,2018-01-31,2024-03-31,75,0.00
6,DEUPROMANMISMEI,fred,"Production, Sales, Work Started and Orders: Pr...",Index 2015=100,M,2018-01-31,2024-01-31,73,0.00
7,DEXUSEU,fred,U.S. Dollars to Euro Spot Exchange Rate,U.S. Dollars to One Euro,M,2018-01-31,2026-05-31,101,0.00
8,ELEC.PRICE.US-ALL.M,eia,all sectors,cents per kilowatt-hour,M,2001-01-31,2026-02-28,302,0.00
9,EXR.D.CHF.EUR.SP00.A,ecb,Swiss franc/Euro ECB reference exchange rate,CHF,M,2018-01-31,2026-05-31,101,0.00


## 5. Correlation Map

Find stable lagged market relationships for commodity-group `material_per_unit`, `hkvar_per_unit`, and `db2_margin`.

In [12]:
correlation_map = build_correlation_map(
    internal_timeseries_df=internal_timeseries,
    market_series_df=market_series,
)
correlation_path = write_json_artifact(correlation_map, output_path=OUTPUT_DIR / "correlation_map.json")

print(f"Wrote {correlation_path}")
pd.DataFrame(
    [
        {"commodity_group": key, "surviving_series": len(value)}
        for key, value in correlation_map.items()
    ]
)

Wrote /Users/dharmendersingh/Documents/Scherzinger_new/notebooks/output/correlation_map.json


,commodity_group,surviving_series
0,BKAES,0
1,BKAGG,2
2,BKAIZ,3
3,MBDIV,0
4,OFRLMG,0
5,OFRSCR,0
6,SOPU,0
7,SOPUZK,0
8,unknown,0


## 6. Forecast Baseline

Build the 12-month commodity-group `db2_margin` forecast baseline. The validation gate follows the plan: at least 50% of eligible commodity groups should have walk-forward MAPE at or below 8%.

In [13]:
forecast_baseline = build_forecast_baseline(
    internal_timeseries_df=internal_timeseries,
    grain="commodity_group",
    metric="db2_margin",
    holdout_months=18,
    horizon_months=12,
)
forecast_path = write_json_artifact(forecast_baseline, output_path=OUTPUT_DIR / "forecast_baseline.json")

forecast_rows = [
    {
        "commodity_group": item["key"],
        "data_months": item["data_months"],
        "eligible": item["eligible_for_acceptance"],
        "winner": item["winner"]["model_name"],
        "mape": item["winner"]["mape"],
    }
    for item in forecast_baseline["series"]
]

print(f"Wrote {forecast_path}")
display(pd.DataFrame([forecast_baseline["validation_summary"]]))
display(pd.DataFrame(forecast_rows))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning

Wrote /Users/dharmendersingh/Documents/Scherzinger_new/notebooks/output/forecast_baseline.json


,eligible_series_count,sparse_series_count,target_mape_pct,meeting_target_count,meeting_target_share
0,3,6,8.0,2,0.666667


,commodity_group,data_months,eligible,winner,mape
0,BKAES,48,True,ses,2.717267
1,BKAGG,48,True,ses,7.972394
2,BKAIZ,42,True,ses,10.420579
3,MBDIV,12,False,ses,69.304755
4,OFRLMG,1,False,ema,inf
5,OFRSCR,2,False,ema,inf
6,SOPU,7,False,ema,101.773008
7,SOPUZK,6,False,ses,15.062976
8,unknown,2,False,ema,inf


## 7. Validation Report

Write a compact Markdown report with the current pass/fail state and the known remaining gap.

In [14]:
summary = forecast_baseline["validation_summary"]
passes_accuracy_gate = (summary["meeting_target_share"] or 0) >= 0.5
report = f"""# Forecasting Notebook Validation Report

Generated from `notebooks/forecasting_market_scenarios.ipynb`.

## Artifacts

- `internal_timeseries.parquet`: written
- `market_series.parquet`: written
- `market_series_catalog.csv`: written
- `correlation_map.json`: written
- `forecast_baseline.json`: written

## Forecast Accuracy Gate

- Metric: `db2_margin`
- Grain: `commodity_group`
- Eligible series: {summary['eligible_series_count']}
- Sparse fallback series: {summary['sparse_series_count']}
- Target MAPE: <= {summary['target_mape_pct']}%
- Meeting target: {summary['meeting_target_count']}
- Meeting target share: {summary['meeting_target_share']:.2%}
- Pass: {passes_accuracy_gate}

## Known Remaining Gap

`BKAIZ` remains above the 8% MAPE threshold on the current history, while the overall eligible-group acceptance gate passes.
"""
report_path = OUTPUT_DIR / "validation_report.md"
report_path.write_text(report)
print(f"Wrote {report_path}")
print(report)

Wrote /Users/dharmendersingh/Documents/Scherzinger_new/notebooks/output/validation_report.md
# Forecasting Notebook Validation Report

Generated from `notebooks/forecasting_market_scenarios.ipynb`.

## Artifacts

- `internal_timeseries.parquet`: written
- `market_series.parquet`: written
- `market_series_catalog.csv`: written
- `correlation_map.json`: written
- `forecast_baseline.json`: written

## Forecast Accuracy Gate

- Metric: `db2_margin`
- Grain: `commodity_group`
- Eligible series: 3
- Sparse fallback series: 6
- Target MAPE: <= 8.0%
- Meeting target: 2
- Meeting target share: 66.67%
- Pass: True

## Known Remaining Gap

`BKAIZ` remains above the 8% MAPE threshold on the current history, while the overall eligible-group acceptance gate passes.



## 8. Multi-grain, multi-metric forecasts

Extend the baseline (commodity-group `db2_margin`) to:
- metrics: `revenue`, `quantity`, `db2_margin`
- grains: `commodity_group`, `business_unit`, top-50 customers + other, top-100 SKUs + other
- cadence: monthly (12m) + quarterly (8q) via Monte Carlo aggregation of monthly samples
- hierarchical bottom-up reconciliation so customer / SKU forecasts sum to commodity-group totals


In [ ]:
from backend.services.forecasting_extensions import (
    build_extended_internal_timeseries,
    build_multi_grain_forecasts,
    build_quarterly_forecasts,
    reconcile_to_parent,
    summarise_forecast_accuracy,
    train_churn_model,
    score_customers_with_churn_model,
    train_quote_conversion_model,
    pipeline_forecast,
    build_churn_labels,
    train_revenue_decline_model,
    score_customers_with_revenue_decline_model,
    persist_forecast_vintage,
    evaluate_forecast_drift,
)
from pathlib import Path as _Path

extended_ts = build_extended_internal_timeseries(
    datasets["invoices_df"],
    customer_names=datasets["customers_df"][["customer_id", "name"]],
)
extended_path = OUTPUT_DIR / "extended_internal_timeseries.parquet"
extended_ts.to_parquet(extended_path, index=False)
print(f"Wrote {extended_path}")
print(extended_ts.groupby(['grain', 'metric']).size().unstack(fill_value=0).head(10))


In [ ]:
monthly_results = build_multi_grain_forecasts(
    extended_ts,
    metrics=("revenue", "quantity", "db2_margin"),
    grains=("commodity_group", "business_unit", "customer", "article"),
    horizon_months=12,
    holdout_months=12,
    min_history_months=6,
    market_series_df=market_series,
    correlation_map=correlation_map,
)

import pandas as pd
accuracy_summary = summarise_forecast_accuracy(
    monthly_results["__accuracy__"],
    target_mape_ratio=12.0,
    target_wape_monetary=25.0,
)
ps = accuracy_summary.get('primary_meeting_share')
print('Primary-tier share meeting target:',
      f"{ps:.2%}" if ps is not None else 'n/a',
      f"(across {accuracy_summary.get('primary_eligible_count', 0)} aggregate series)")
display(pd.DataFrame(accuracy_summary['by_metric_grain']))

# Show which series were upgraded by SARIMAX-with-exog
exog_used = [r for r in monthly_results['__accuracy__'] if r.get('uses_exog')]
if exog_used:
    print(f"\nSARIMAX-with-exog upgraded {len(exog_used)} series:")
    display(pd.DataFrame(exog_used)[['metric','grain','key','model','mape','wape']])


In [ ]:
import numpy as np
quarterly_results = build_quarterly_forecasts(monthly_results, quarters=8)
print('Quarterly horizons built for metrics:', [m for m in quarterly_results.keys() if not m.startswith('__')])
example = quarterly_results['revenue']['commodity_group']
example_key = next(iter(example))
print(f'Example commodity_group={example_key} quarterly revenue forecast:')
display(pd.DataFrame(example[example_key]))


In [ ]:
reconcile_customer = reconcile_to_parent(
    monthly_results,
    metric='revenue',
    child_grain='customer',
    parent_grain='commodity_group',
    invoices_df=datasets['invoices_df'],
)
reconcile_article = reconcile_to_parent(
    monthly_results,
    metric='revenue',
    child_grain='article',
    parent_grain='commodity_group',
    invoices_df=datasets['invoices_df'],
)

residuals_df = pd.DataFrame(reconcile_customer['residuals'] + reconcile_article['residuals'])
print(f"Customer-level series reconciled: {len(reconcile_customer['reconciled'])}")
print(f"Article-level series reconciled: {len(reconcile_article['reconciled'])}")
display(residuals_df.sort_values('pre_reconciliation_residual_pct', ascending=False).head(15))


### Write multi-grain forecast artifacts


In [ ]:
def _stringify_ts(forecast_rows):
    out = []
    for row in forecast_rows:
        new = dict(row)
        if 'ts' in new:
            new['ts'] = pd.Timestamp(new['ts']).isoformat()
        out.append(new)
    return out

def _serialize_metric_results(metric_results):
    payload = {}
    for grain, series_map in metric_results.items():
        payload[grain] = {
            key: {
                'data_months': item['data_months'],
                'eligible': item['eligible_for_acceptance'],
                'winner_model': item['winner']['model_name'],
                'winner_mape': item['winner']['mape'],
                'forecast': _stringify_ts(item['forecast']),
            }
            for key, item in series_map.items()
        }
    return payload

revenue_payload = _serialize_metric_results(monthly_results['revenue'])
quantity_payload = _serialize_metric_results(monthly_results['quantity'])
margin_payload = _serialize_metric_results(monthly_results['db2_margin'])

write_json_artifact(revenue_payload, output_path=OUTPUT_DIR / 'forecast_revenue.json')
write_json_artifact(quantity_payload, output_path=OUTPUT_DIR / 'forecast_quantity.json')
write_json_artifact(margin_payload, output_path=OUTPUT_DIR / 'forecast_margin.json')

quarterly_payload = {}
for metric, grain_map in quarterly_results.items():
    quarterly_payload[metric] = {}
    for grain, series_map in grain_map.items():
        quarterly_payload[metric][grain] = {
            key: _stringify_ts(rows) for key, rows in series_map.items()
        }
write_json_artifact(quarterly_payload, output_path=OUTPUT_DIR / 'forecast_quarterly.json')
print('Wrote revenue/quantity/margin/quarterly forecast JSONs')

# M11: persist this forecast vintage for future drift analysis
vintage_path = persist_forecast_vintage(
    monthly_results, output_path=OUTPUT_DIR / 'forecast_vintages.parquet',
)
print(f'M11: appended this vintage to {vintage_path}')

# If we have prior vintages, compute drift against current actuals
drift = evaluate_forecast_drift(vintage_path, extended_timeseries=extended_ts)
if not drift.empty:
    print('M11: drift summary (prior vintages vs current actuals)')
    display(drift)


In [ ]:
# Per-SKU table (long form) covering monthly + quarterly
sku_rows = []
sku_monthly = monthly_results['revenue'].get('article', {})
sku_quarterly = quarterly_results['revenue'].get('article', {})
for key, summary in sku_monthly.items():
    method = 'direct' if not key.startswith('other__') else 'proportional_bucket'
    for row in summary['forecast']:
        sku_rows.append({
            'article_key': key,
            'cadence': 'monthly',
            'ts': row['ts'],
            'p50': row['p50'],
            'p80_low': row['p80_low'], 'p80_high': row['p80_high'],
            'p95_low': row['p95_low'], 'p95_high': row['p95_high'],
            'forecast_method': method,
        })
for key, rows in sku_quarterly.items():
    method = 'direct' if not key.startswith('other__') else 'proportional_bucket'
    for row in rows:
        sku_rows.append({
            'article_key': key,
            'cadence': 'quarterly',
            'ts': row['ts'],
            'p50': row['p50'],
            'p80_low': row['p80_low'], 'p80_high': row['p80_high'],
            'p95_low': row['p95_low'], 'p95_high': row['p95_high'],
            'forecast_method': method,
        })
sku_df = pd.DataFrame(sku_rows)
sku_path = OUTPUT_DIR / 'sku_forecasts.parquet'
sku_df['ts'] = pd.to_datetime(sku_df['ts'])
sku_df.to_parquet(sku_path, index=False)
print(f'Wrote {sku_path}  rows={len(sku_df)}')
display(sku_df.head())


In [ ]:
# Per-customer table
cust_rows = []
cust_monthly = monthly_results['revenue'].get('customer', {})
cust_quarterly = quarterly_results['revenue'].get('customer', {})
name_map = datasets['customers_df'].set_index('customer_id')['name'].to_dict()
for key, summary in cust_monthly.items():
    display_name = name_map.get(key) if not key.startswith('other__') else None
    for row in summary['forecast']:
        cust_rows.append({
            'customer_key': key,
            'display_name': display_name,
            'cadence': 'monthly',
            'ts': row['ts'],
            'p50': row['p50'],
            'p80_low': row['p80_low'], 'p80_high': row['p80_high'],
            'p95_low': row['p95_low'], 'p95_high': row['p95_high'],
        })
for key, rows in cust_quarterly.items():
    display_name = name_map.get(key) if not key.startswith('other__') else None
    for row in rows:
        cust_rows.append({
            'customer_key': key,
            'display_name': display_name,
            'cadence': 'quarterly',
            'ts': row['ts'],
            'p50': row['p50'],
            'p80_low': row['p80_low'], 'p80_high': row['p80_high'],
            'p95_low': row['p95_low'], 'p95_high': row['p95_high'],
        })
cust_df = pd.DataFrame(cust_rows)
cust_df['ts'] = pd.to_datetime(cust_df['ts'])
cust_path = OUTPUT_DIR / 'customer_forecasts.parquet'
cust_df.to_parquet(cust_path, index=False)
print(f'Wrote {cust_path}  rows={len(cust_df)}')
display(cust_df.head())


## 9. Customer churn model

Predict probability each currently-active customer goes inactive over the next 1Q / 2Q / 4Q windows. Definition: no invoice in trailing 6 months **and** no won quote in trailing 3 months at horizon end. Trained with rolling as-of dates to keep labels honest (no peeking past horizon end).

In [ ]:
import datetime as _dt

# Score as of the most recent month-end in the invoice data
as_of = pd.to_datetime(datasets['invoices_df']['date']).max().normalize() + pd.offsets.MonthEnd(0)
print(f'Scoring churn as of: {as_of}')

# Build revenue-by-customer forecast dict (top-50 directly forecast)
revenue_forecast_by_customer = {
    key: summary['forecast']
    for key, summary in monthly_results.get('revenue', {}).get('customer', {}).items()
    if not key.startswith('other__')
}

churn_outputs = []
for horizon in (3, 6, 12):
    artifact = train_churn_model(
        datasets['invoices_df'],
        datasets['quotes_df'],
        score_as_of=as_of,
        horizon_months=horizon,
        lookback_label_dates=3,
    )
    scored = score_customers_with_churn_model(
        artifact,
        datasets['invoices_df'],
        datasets['quotes_df'],
        revenue_forecast_by_customer=revenue_forecast_by_customer,
        customer_names=datasets['customers_df'],
    )
    scored['horizon_q'] = horizon // 3
    churn_outputs.append((horizon, artifact, scored))
    print(f'  horizon={horizon}m  AUC={artifact.metrics["auc"]:.3f}  '
          f'AP={artifact.metrics["average_precision"]:.3f}  '
          f'Brier={artifact.metrics["brier"]:.3f}  '
          f'customers_scored={len(scored)}')


In [ ]:
# Pivot churn predictions into one row per customer with p_churn_1q / 2q / 4q
pivot = None
for horizon, artifact, scored in churn_outputs:
    col = f'p_churn_{horizon // 3}q'
    risk_col = f'at_risk_revenue_{horizon // 3}q'
    sub = scored[['customer_id', 'p_churn', 'at_risk_revenue']].rename(
        columns={'p_churn': col, 'at_risk_revenue': risk_col}
    )
    pivot = sub if pivot is None else pivot.merge(sub, on='customer_id', how='outer')

pivot = pivot.merge(
    datasets['customers_df'][['customer_id', 'name']],
    on='customer_id',
    how='left',
)

# Top SKUs per customer (last 12 months)
recent_cutoff = as_of - pd.DateOffset(months=12)
inv_recent = datasets['invoices_df'][pd.to_datetime(datasets['invoices_df']['date']) >= recent_cutoff]
top_skus_per_cust = (
    inv_recent.groupby(['customer_id', 'article_id'])['revenue'].sum()
    .reset_index()
    .sort_values(['customer_id', 'revenue'], ascending=[True, False])
    .groupby('customer_id')['article_id']
    .apply(lambda s: ','.join(s.head(5).astype(str)))
    .reset_index()
    .rename(columns={'article_id': 'top_skus_12m'})
)
pivot = pivot.merge(top_skus_per_cust, on='customer_id', how='left')

pivot = pivot.sort_values('p_churn_4q', ascending=False)

# M8: revenue-decline companion model (predicts wallet erosion that the
# binary churn label misses)
decline_artifact = train_revenue_decline_model(
    datasets['invoices_df'], datasets['quotes_df'],
    score_as_of=as_of, horizon_months=12, decline_threshold=0.5,
)
print(f'M8 revenue-decline model: AUC={decline_artifact.metrics["auc"]:.3f}  '
      f'AP={decline_artifact.metrics["average_precision"]:.3f}  '
      f'Brier={decline_artifact.metrics["brier"]:.3f}')
decline_scored = score_customers_with_revenue_decline_model(
    decline_artifact, datasets['invoices_df'], datasets['quotes_df'],
)
pivot = pivot.merge(
    decline_scored[['customer_id', 'p_major_decline']],
    on='customer_id', how='left',
)

churn_csv = OUTPUT_DIR / 'churn_predictions.csv'
pivot.to_csv(churn_csv, index=False)
print(f'Wrote {churn_csv}  rows={len(pivot)}')
display(pivot.head(15))


## 10. Quote-to-order conversion + open-pipeline forecast

Train a win-rate model on historical quotes (`is_won`), apply to any open quotes to estimate expected booked revenue in the next 1-2 quarters.


In [ ]:
# Train conversion model on historical quotes with known outcome
conv_artifact = train_quote_conversion_model(datasets['quotes_df'])
print('Quote conversion metrics:', conv_artifact.metrics)

# Identify open quotes (status not yet won/lost) - those without is_won True/False resolved
open_quotes = datasets['quotes_df'].copy()
if 'status' in open_quotes.columns:
    open_quotes = open_quotes[open_quotes['status'].astype(str).str.lower().isin(['open', 'pending', ''])]
    if open_quotes.empty:
        # Fallback: treat last-90-day quotes as proxy for pipeline
        cutoff = pd.to_datetime(datasets['quotes_df']['date']).max() - pd.Timedelta(days=90)
        open_quotes = datasets['quotes_df'][pd.to_datetime(datasets['quotes_df']['date']) >= cutoff]
        print(f'No open status quotes; using last 90 days ({len(open_quotes)} quotes) as proxy pipeline')
else:
    cutoff = pd.to_datetime(datasets['quotes_df']['date']).max() - pd.Timedelta(days=90)
    open_quotes = datasets['quotes_df'][pd.to_datetime(datasets['quotes_df']['date']) >= cutoff]

pipeline_df = pipeline_forecast(conv_artifact, open_quotes)
pipeline_path = OUTPUT_DIR / 'pipeline_forecast.csv'
pipeline_df.to_csv(pipeline_path, index=False)
print(f'Wrote {pipeline_path}  rows={len(pipeline_df)}')
if not pipeline_df.empty:
    display(pipeline_df.head(15))
    by_quarter = pipeline_df.groupby('quarter')['expected_revenue'].sum().reset_index()
    print('Expected booked revenue by quarter:')
    display(by_quarter)


## 11. Extended validation report


In [ ]:
from datetime import datetime, timezone

primary_rows = [r for r in accuracy_summary['by_metric_grain'] if r['gate_tier'] == 'primary']
info_rows = [r for r in accuracy_summary['by_metric_grain'] if r['gate_tier'] == 'informational']
def _fmt_rows(rows):
    return '\n'.join(
        f"| {r['metric']} | {r['grain']} | {r['error_metric']} | {r['target_pct']:.0f}% | {r['eligible_series']} | {r['median_error']:.2f} | {r['meeting_share']:.0%} |"
        for r in rows
    ) or '| (none) | | | | | | |'
primary_table = _fmt_rows(primary_rows)
info_table = _fmt_rows(info_rows)

churn_table_rows = []
for horizon, art, _scored in churn_outputs:
    m = art.metrics
    churn_table_rows.append(
        f"| {horizon}m ({horizon // 3}Q) | {m['auc']:.3f} | {m['average_precision']:.3f} | {m['brier']:.3f} |"
    )

residual_rows = []
for r in (reconcile_customer['residuals'] + reconcile_article['residuals']):
    residual_rows.append(
        f"| {r['parent']} | {r['child_grain']} | {r['n_children']} | {r['pre_reconciliation_residual_pct']:.2f}% |"
    )

passes_main_gate = (forecast_baseline['validation_summary']['meeting_target_share'] or 0) >= 0.5
primary_share = accuracy_summary['primary_meeting_share'] or 0.0
primary_count = accuracy_summary.get('primary_eligible_count', 0)
passes_extended_gate = primary_share >= 0.6

ext_report = f"""# Forecasting Notebook Validation Report

Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}

## Artifacts written

- `internal_timeseries.parquet`, `extended_internal_timeseries.parquet`
- `market_series.parquet`, `market_series_catalog.csv`, `correlation_map.json`
- `forecast_baseline.json` (commodity_group x db2_margin, monthly)
- `forecast_revenue.json`, `forecast_quantity.json`, `forecast_margin.json` (monthly, all grains)
- `forecast_quarterly.json` (all metrics, all grains, 8q horizon)
- `sku_forecasts.parquet`, `customer_forecasts.parquet`
- `churn_predictions.csv`, `pipeline_forecast.csv`
- `summary.html`

## Baseline accuracy gate (db2_margin x commodity_group)

- Eligible series: {forecast_baseline['validation_summary']['eligible_series_count']}
- Target MAPE: <= {forecast_baseline['validation_summary']['target_mape_pct']}%
- Meeting target share: {forecast_baseline['validation_summary']['meeting_target_share']:.2%}
- Pass (>= 50%): **{passes_main_gate}**

## Multi-grain forecast accuracy

Two tiers:

- **Primary gate** (aggregate grains: total / business_unit / commodity_group): the levels finance and sales plan against. Ratio metrics use MAPE <=12%, monetary metrics use WAPE <=25%.
- **Informational** (customer / article grains): per-account and per-SKU monthly cadence is inherently bursty in industrial B2B (long order cycles, few invoices/month per account). These forecasts are useful for ranking and at-risk-revenue calculations but should not be consumed as point forecasts. Aggregate quarterly views are more reliable - see `forecast_quarterly.json`.

### Primary gate

| Metric | Grain | Error | Target | Eligible series | Median % | Meeting target |
|--------|-------|-------|--------|-----------------|----------|----------------|
{primary_table}

Primary gate: {primary_count} eligible series, **{primary_share:.2%}** meeting target -> Pass (>=60%): **{passes_extended_gate}**

### Informational (customer / article grains)

| Metric | Grain | Error | Target | Eligible series | Median % | Meeting target |
|--------|-------|-------|--------|-----------------|----------|----------------|
{info_table}

## Churn model performance (time-series CV)

| Horizon | AUC | Avg. Precision | Brier |
|---------|-----|----------------|-------|
{chr(10).join(churn_table_rows)}

Target AUC >= 0.70. Customers scored: top-{len(pivot)}.

## Quote conversion model

- AUC: {conv_artifact.metrics['auc']:.3f}
- Average precision: {conv_artifact.metrics['average_precision']:.3f}
- Brier: {conv_artifact.metrics['brier']:.3f}
- Pipeline forecast rows: {len(pipeline_df)}

## Hierarchical reconciliation residuals (pre-scaling, p50 sums)

| Parent | Child grain | Children | Residual % |
|--------|-------------|----------|------------|
{chr(10).join(residual_rows) if residual_rows else '| (no reconciled groups) | | | |'}

## Known gaps

- Long-tail SKUs (>100 by rank) are forecast via proportional bucket roll-down rather than directly.
- Customer names anonymised in the input data; churn predictions are by customer_id only.

## What changed in this revision

- Monetary metrics (revenue/quantity) now scored via WAPE, the standard
  metric for bursty positive demand series. Target tightened to <=25%.
- SARIMAX-with-exog model added for commodity_group series with strong
  leading-indicator correlations. Each upgraded series is annotated in the
  per-series JSON with the macro `exog_series_ids` used.
"""

report_path = OUTPUT_DIR / 'validation_report.md'
report_path.write_text(ext_report)
print(f'Wrote {report_path}')
print(ext_report[:1200])


## 12. Client summary HTML


In [ ]:
import html as _html

top_churn = pivot.head(20)[['customer_id', 'name', 'p_churn_4q', 'at_risk_revenue_4q', 'top_skus_12m']].fillna('')
top_pipeline_q = (
    pipeline_df.groupby('quarter')['expected_revenue'].sum().reset_index()
    if not pipeline_df.empty else pd.DataFrame(columns=['quarter', 'expected_revenue'])
)

# Revenue by commodity_group quarterly view
rev_q = quarterly_results['revenue']['commodity_group']
quarterly_rev_rows = []
for group, rows in rev_q.items():
    for row in rows:
        quarterly_rev_rows.append({
            'commodity_group': group,
            'quarter': pd.Timestamp(row['ts']).strftime('%Y Q%q').replace('%q', str(pd.Timestamp(row['ts']).quarter)),
            'p50_revenue': row['p50'],
            'p80_low': row['p80_low'],
            'p80_high': row['p80_high'],
        })
quarterly_rev_df = pd.DataFrame(quarterly_rev_rows)

def df_to_html(df, fmt=None):
    if df.empty:
        return '<p><em>No data</em></p>'
    if fmt:
        return df.to_html(index=False, float_format=fmt, escape=True)
    return df.to_html(index=False, escape=True)

html = f"""<!doctype html>
<html><head><meta charset='utf-8'><title>Scherzinger Forecast + Churn Summary</title>
<style>
body {{ font-family: -apple-system, system-ui, sans-serif; max-width: 1080px; margin: 2rem auto; color: #1a1a1a; }}
h1, h2 {{ color: #0e3b5c; }}
table {{ border-collapse: collapse; margin: 1rem 0; width: 100%; }}
th, td {{ padding: 6px 10px; border-bottom: 1px solid #e5e7eb; text-align: right; }}
th {{ background: #f8fafc; text-align: left; }}
td:first-child, th:first-child {{ text-align: left; }}
.kpi {{ display: inline-block; padding: 1rem 1.5rem; margin: 0.25rem; background: #f1f5f9; border-radius: 8px; }}
.kpi .v {{ font-size: 1.4rem; font-weight: 700; color: #0e3b5c; }}
.kpi .l {{ font-size: 0.85rem; color: #475569; }}
</style></head><body>
<h1>Scherzinger - Forecast &amp; Churn Summary</h1>
<p>Generated {pd.Timestamp.utcnow().strftime('%Y-%m-%d %H:%M UTC')}. Scoring as-of {as_of.strftime('%Y-%m-%d')}.</p>

<div>
  <div class='kpi'><div class='v'>{accuracy_summary['overall_share_meeting_target']:.0%}</div><div class='l'>Series meeting &lt;=12% MAPE</div></div>
  <div class='kpi'><div class='v'>{len(pivot)}</div><div class='l'>Customers scored for churn</div></div>
  <div class='kpi'><div class='v'>{int(pivot['p_churn_4q'].ge(0.5).sum())}</div><div class='l'>Customers with &gt;=50% 4Q churn prob</div></div>
  <div class='kpi'><div class='v'>EUR {pivot['at_risk_revenue_4q'].fillna(0).sum():,.0f}</div><div class='l'>Total 4Q at-risk revenue</div></div>
</div>

<h2>Top 20 churn-risk customers (4Q horizon)</h2>
{df_to_html(top_churn, fmt=lambda v: f'{v:,.3f}' if isinstance(v, float) else str(v))}

<h2>Quarterly revenue forecast by commodity group (P50, EUR)</h2>
{df_to_html(quarterly_rev_df, fmt=lambda v: f'{v:,.0f}' if isinstance(v, float) else str(v))}

<h2>Open-pipeline expected booked revenue by quarter</h2>
{df_to_html(top_pipeline_q, fmt=lambda v: f'{v:,.0f}' if isinstance(v, float) else str(v))}

<h2>Multi-grain forecast accuracy</h2>
{df_to_html(pd.DataFrame(accuracy_summary['by_metric_grain']), fmt=lambda v: f'{v:.2f}' if isinstance(v, float) else str(v))}
</body></html>
"""

(OUTPUT_DIR / 'summary.html').write_text(html)
print(f"Wrote {OUTPUT_DIR / 'summary.html'}")
